# Chapter 1 · How a Model Learns

A straight line that learns to estimate fictitious house prices by correcting its two parameters with gradient descent. Standard Python and CPU: there is nothing to install. Run the cells in order.

The full explanation is in the PDF chapter. This notebook is the lab: the same program, in parts, for you to run and experiment with.

## 1. The data and the prediction

Three training houses and one held-out house. `to_input` counts the area in blocks of 50 m², and `predict` applies the rule ŷ = wx + b, in thousands of euros.

In [1]:
"""Chapter 1: fictitious house prices. Standard Python and CPU."""
import math

# Area in m²; price in thousands of euros.
HOUSES = [(50.0, 150.0), (100.0, 250.0), (150.0, 350.0)]
HELD_OUT = (75.0, 200.0)


def to_input(area):
    return area / 50.0


def predict(area, w, b):
    return w * to_input(area) + b

In [2]:
print("predict(100, 50, 50) =", predict(100, 50, 50))
for area, price in HOUSES:
    estimate = predict(area, 50, 50)
    print(f"{area:g} m²: price {price:g}, estimate {estimate:g}, error {estimate - price:g}")

predict(100, 50, 50) = 150.0
50 m²: price 150, estimate 100, error -50
100 m²: price 250, estimate 150, error -100
150 m²: price 350, estimate 200, error -150


## 2. The loss and the gradients

`loss` is the mean squared error. `gradients` adds up the contribution of each house: `2 * error * x` for the weight and `2 * error` for the bias, and divides by the number of houses.

In [3]:
def loss(houses, w, b):
    errors = [predict(a, w, b) - price for a, price in houses]
    return sum(e ** 2 for e in errors) / len(errors)


def gradients(houses, w, b):
    dw, db = 0.0, 0.0
    for area, price in houses:
        x = to_input(area)
        error = w * x + b - price
        dw += 2 * error * x
        db += 2 * error
    n = len(houses)
    return dw / n, db / n

Check the chapter's numbers: the initial loss, the loss with w = 60 and the trials with w = 49 and w = 51. The difference between these last two, divided by 2, matches the gradient of the weight.

In [4]:
for w in (50, 60, 49, 51):
    print(f"w = {w}, b = 50: loss {loss(HOUSES, w, 50):.2f}")
dw, db = gradients(HOUSES, 50, 50)
print(f"Gradients at the starting point: weight {dw:.2f}, bias {db:.2f}")
print(f"(L(51) - L(49)) / 2 = {(loss(HOUSES, 51, 50) - loss(HOUSES, 49, 50)) / 2:.2f}")

w = 50, b = 50: loss 11666.67
w = 60, b = 50: loss 7466.67
w = 49, b = 50: loss 12138.00
w = 51, b = 50: loss 11204.67
Gradients at the starting point: weight -466.67, bias -200.00
(L(51) - L(49)) / 2 = -466.67


## 3. One step by hand

One update with rate 0.05: both gradients are computed with the old parameters, and only then are both parameters updated.

In [5]:
rate = 0.05
w, b = 50.0, 50.0
dw, db = gradients(HOUSES, w, b)
w, b = w - rate * dw, b - rate * db
print(f"After one step: w = {w:.2f}, b = {b:.2f}, loss {loss(HOUSES, w, b):.2f}")

After one step: w = 73.33, b = 60.00, loss 2351.85


## 4. Training

`train` repeats the step and stores in the history the number of updates, the weight, the bias and the loss.

In [6]:
def train(houses, rate=0.05, steps=1000):
    w, b = 50.0, 50.0
    history = [(0, w, b, loss(houses, w, b))]
    for step in range(1, steps + 1):
        dw, db = gradients(houses, w, b)
        w, b = w - rate * dw, b - rate * db
        if not all(math.isfinite(v) and abs(v) < 1e100 for v in (w, b)):
            raise ValueError("Training diverges: lower the rate.")
        history.append((step, w, b, loss(houses, w, b)))
    return w, b, history

## 5. The complete experiment

`compare_rates` trains from the same point with three rates and twenty steps. `run` trains for 1000 steps, estimates the held-out house and compares the rates.

In [7]:
def compare_rates(rates=(0.005, 0.05, 0.3), steps=20):
    for rate in rates:
        _, _, history = train(HOUSES, rate=rate, steps=steps)
        print(f"Rate {rate}: loss after {steps} steps {history[-1][3]:.6g}")


def run():
    w, b, history = train(HOUSES)
    print("Step    Weight      Bias  Loss")
    for step, wp, bp, value in history:
        if step in (0, 1, 2, 20, 100, 1000):
            print(f"{step:4d} {wp:9.4f} {bp:9.4f}  {value:.6g}")
    area, price = HELD_OUT
    print(f"Held-out house of {area:g} m²: estimate "
          f"{predict(area, w, b):.2f}, price {price:g} (thousands of euros)")
    compare_rates()
    return history

In [8]:
history = run()

Step    Weight      Bias  Loss
   0   50.0000   50.0000  11666.7
   1   73.3333   60.0000  2351.85
   2   83.7778   64.3333  503.453
  20   93.6347   64.4699  30.0364
 100   97.5807   55.4996  4.33887
1000  100.0000   50.0001  1.52871e-09
Held-out house of 75 m²: estimate 200.00, price 200 (thousands of euros)
Rate 0.005: loss after 20 steps 1231.81
Rate 0.05: loss after 20 steps 30.0364
Rate 0.3: loss after 20 steps 5.54075e+18


## 6. Experiment with the rate

Try other rates with twenty steps (exercise 9): 0.1, 0.15, 0.2… Write down first what you expect to happen. If a rate makes the parameters blow up, the program stops with a warning.

In [9]:
compare_rates(rates=(0.1, 0.15, 0.2))

Rate 0.1: loss after 20 steps 18.4081
Rate 0.15: loss after 20 steps 11.2142
Rate 0.2: loss after 20 steps 3.15795e+07


## Exercises

1. Explain in your own words what the weight and the bias mean in the example. What is the difference between changing the area of a house and changing the weight of the model?

2. A house has 125 m². Convert its area into the input x and compute its estimated price with w=50, b=50. Express the result in euros.

3. Reconstruct the three initial errors and their mean loss. Explain why the loss is not expressed directly in euros.

4. Increase only the weight, from 50 to 60, and keep the bias. Compute the three predictions and check whether the loss improves.

5. Compute each house's contribution 2eᵢxᵢ to the gradient of the weight at the starting point. Do the same with the contributions 2eᵢ to the gradient of the bias, and find the averages.

6. Apply one update with rate 0.05. Interpret the new weight and bias and compute the resulting loss.

7. Why can the bias increase in the first step even though the data were created with a bias of 50?

8. Run the program and check the three rates in the chapter's table. Explain the difference between making slow progress and diverging.

9. Try the rates 0.1, 0.15 and 0.2 with twenty steps, in the notebook or by changing `compare_rates`. Which one lowers the loss the most? At what rate does training start to diverge?

10. Estimate the price of the 75 m² held-out house using the final parameters. Is it interpolation or extrapolation? What are the limits of this check?

11. Explain the difference between training and inference using the chapter's code.

12. Two houses have 100 m², but one costs 200,000 euros and the other 300,000. Can this model get both exactly right? Reason about what information might be missing.

13. Describe in 100–150 words what this example shares with training a language model, and mention at least three differences.

## References

Inspiration: the [first lecture](https://www.youtube.com/watch?v=VMj-3S1tku0) of Andrej Karpathy's [Zero to Hero](https://github.com/karpathy/nn-zero-to-hero), with its derivative engine [micrograd](https://github.com/karpathy/micrograd). The house data, code and explanations are original work.